# Perron: Google Gemma 4 E2B Empirical GPU Pointwise Inference
### Full Evaluation Across All 344 SWE-bench Verified Tasks (19,298 Prompts)
**Author**: Sultan Haikal ([GitHub: @yvliet](https://github.com/yvliet/perron))  
**Model**: `google/gemma-4-e2b-it` (Version 1 on Kaggle Models)  
**Hardware Target**: Single NVIDIA Tesla T4 (16GB VRAM)  
**Protocol Invariant**: PREREG-GEMMA4-POINTWISE-v1 (Locked P0 Variant, Yes/No Log-Odds Scoring)  

This notebook executes live GPU forward passes over all 19,298 pre-rendered candidate prompts to evaluate the pointwise ranking capability of Google Gemma 4 E2B.


In [ ]:
# Step 1: Environment Setup, CUDA Diagnostics & Hugging Face Authentication
import os
import sys
import time
import json
import re
import gc
import shutil
from pathlib import Path
import numpy as np
import pandas as pd
import torch

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Authenticate with Hugging Face via Kaggle Secrets if available
try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    hf_token = secrets.get_secret('HF_TOKEN')
    if hf_token:
        os.environ['HF_TOKEN'] = hf_token
        os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
        print('[+] Successfully attached HF_TOKEN from Kaggle Secrets.')
except Exception:
    print('[-] No Kaggle Secrets found. Proceeding with public/local models.')

print(f'Python Version : {sys.version.split()[0]}')
print(f'PyTorch Version: {torch.__version__}')
cuda_available = torch.cuda.is_available()
gpu_count = torch.cuda.device_count() if cuda_available else 0
print(f'CUDA Available : {cuda_available} (GPU Count: {gpu_count})')

if cuda_available:
    for i in range(gpu_count):
        p = torch.cuda.get_device_properties(i)
        print(f'  [GPU {i}]: {p.name} | Total VRAM: {p.total_memory / (1024**3):.2f} GB')
else:
    print('WARNING: CUDA is not available. GPU acceleration is required for full-run throughput.')


In [ ]:
# Step 2: Ensure Dependencies
!pip install -q accelerate bitsandbytes scipy transformers


In [ ]:
# Step 3: Ingest Pre-Rendered Evaluation Prompts
candidate_paths = [
    Path('/kaggle/input/perron-gemma4-benchmark/gemma4_e2b_prompts_19298.parquet'),
    Path('/kaggle/input/datasets/yvliet/perron-gemma4-benchmark/gemma4_e2b_prompts_19298.parquet'),
    Path('data/gemma4_e2b_prompts_19298.parquet'),
]

prompts_path = None
for cp in candidate_paths:
    if cp.exists():
        prompts_path = cp
        break

if prompts_path is None:
    base_in = Path('/kaggle/input')
    if base_in.exists():
        for found in base_in.rglob('*gemma4_e2b_prompts*.parquet'):
            if found.is_file():
                prompts_path = found
                break

if prompts_path is None:
    raise FileNotFoundError('Could not find gemma4_e2b_prompts_19298.parquet in /kaggle/input.')

print(f'[+] Ingesting pre-rendered prompt cohort from: {prompts_path}')
df_prompts = pd.read_parquet(prompts_path)
print(f'[+] Loaded {len(df_prompts)} prompts across {df_prompts["instance_id"].nunique()} tasks.')
print(f'    Columns: {list(df_prompts.columns)}')
print(f'    Gold candidates: {df_prompts["is_gold"].sum()}')


In [ ]:
# Step 4: Resolve & Load Gemma 4 E2B Model & Tokenizer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

TARGET_MODEL_ID = 'google/gemma-4-e2b-it'

def resolve_model_path(model_id: str) -> str:
    if Path(model_id).exists() and (Path(model_id) / 'config.json').exists():
        return model_id
    base_in = Path('/kaggle/input')
    if base_in.exists():
        for p in base_in.rglob('config.json'):
            p_str = str(p.parent).lower()
            if 'e2b' in p_str and 'gemma' in p_str:
                print(f'[+] Resolved local Kaggle model mount: {p.parent}')
                return str(p.parent)
    return model_id

resolved_model_path = resolve_model_path(TARGET_MODEL_ID)
print(f'[+] Loading Gemma 4 E2B from: {resolved_model_path}')

tokenizer = AutoTokenizer.from_pretrained(resolved_model_path, padding_side='left')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Identify Yes / No token IDs for binary log-odds scoring
yes_token_ids, no_token_ids = set(), set()
for word in ['Yes', ' Yes', 'YES', ' yes']:
    enc = tokenizer.encode(word, add_special_tokens=False)
    if enc: yes_token_ids.add(enc[-1])
for word in ['No', ' No', 'NO', ' no']:
    enc = tokenizer.encode(word, add_special_tokens=False)
    if enc: no_token_ids.add(enc[-1])

yes_ids = sorted(list(yes_token_ids))
no_ids = sorted(list(no_token_ids))
print(f'[+] Resolved Token IDs -> Yes: {yes_ids}, No: {no_ids}')

# Configure BitsAndBytes 4-bit NF4 Quantization to guarantee zero CUDA OOM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
) if torch.cuda.is_available() else None

model_kwargs = {
    'quantization_config': bnb_config,
    'device_map': 'auto',
    'trust_remote_code': True,
}

model = AutoModelForCausalLM.from_pretrained(resolved_model_path, **model_kwargs)
model.eval()
print('[+] Model successfully loaded.')
if hasattr(model, 'hf_device_map'):
    print(f'    Device Map: {model.hf_device_map}')


In [ ]:
# Step 5: Low-VRAM Zero-Padding Pointwise Scoring Engine

def format_chat_prompt(raw_prompt: str) -> str:
    messages = [{'role': 'user', 'content': raw_prompt}]
    try:
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except Exception:
        return f'<start_of_turn>user\n{raw_prompt}<end_of_turn>\n<start_of_turn>model\n'

@torch.inference_mode()
def score_single_prompt(raw_prompt: str) -> tuple[float, float, int]:
    """
    Scores a single prompt without padding overhead, minimizing peak VRAM and eliminating attention mask artifacts.
    """
    chat_p = format_chat_prompt(raw_prompt)
    inputs = tokenizer(chat_p, return_tensors='pt', truncation=True, max_length=2048)
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    try:
        outputs = model(**inputs, logits_to_keep=1, use_cache=False)
        logits = outputs.logits[:, -1, :].to(torch.float32)
    except TypeError:
        try:
            outputs = model(**inputs, use_cache=False)
            logits = outputs.logits[:, -1, :].to(torch.float32)
        except TypeError:
            outputs = model(**inputs)
            logits = outputs.logits[:, -1, :].to(torch.float32)

    probs = torch.softmax(logits, dim=-1)
    row_logits = logits[0]
    row_probs = probs[0]

    s_val = float(torch.logsumexp(row_logits[yes_ids], dim=-1) - torch.logsumexp(row_logits[no_ids], dim=-1))
    mass_val = float(row_probs[yes_ids].sum() + row_probs[no_ids].sum())
    n_tok = int(inputs['attention_mask'].sum().item())
    return s_val, mass_val, n_tok

# Pre-flight smoke test probe
print('[+] Running smoke test probe...')
smoke_s, smoke_m, smoke_t = score_single_prompt(
    'Problem: ZeroDivisionError in math.py\nCandidate: def div(a, b): return a / b\nDoes this cause the bug? Answer Yes or No.'
)
print(f'[+] Smoke test passed! Log-odds: {smoke_s:.4f}, mass: {smoke_m:.4f}, tokens: {smoke_t}')


In [ ]:
# Step 6: Evaluation Protocol Runner Across All 19,298 Prompts
OUTPUT_CHECKPOINT = Path('/kaggle/working/gemma4_e2b_checkpoint.parquet')
FINAL_SCORES_PARQUET = Path('/kaggle/working/gemma4_e2b_gpu_scores.parquet')
FINAL_SCORES_STANDARD = Path('/kaggle/working/scores.parquet')
FINAL_METRICS_JSON = Path('/kaggle/working/gemma4_e2b_summary_metrics.json')

completed_records = []
scored_keys = set()
if OUTPUT_CHECKPOINT.exists():
    try:
        df_prev = pd.read_parquet(OUTPUT_CHECKPOINT)
        completed_records = df_prev.to_dict('records')
        scored_keys = set(zip(df_prev['instance_id'], df_prev['candidate_id']))
        print(f'[+] Resuming from checkpoint: {len(completed_records)} prompts already scored.')
    except Exception as e:
        print(f'[-] Checkpoint load failed: {e}. Starting fresh.')

start_time = time.time()
n_total = len(df_prompts)

print(f'[+] Starting pointwise inference loop over {n_total} candidate prompts...')

for idx, row in df_prompts.iterrows():
    key = (row['instance_id'], row['candidate_id'])
    if key in scored_keys:
        continue

    s_val, mass_val, n_tok = score_single_prompt(row['raw_prompt'])
    completed_records.append({
        'instance_id': row['instance_id'],
        'candidate_id': row['candidate_id'],
        'prompt_variant': row.get('prompt_variant', 'P0'),
        's': s_val,
        'format_mass': mass_val,
        'n_prompt_tokens': n_tok,
        'is_gold': row.get('is_gold', 0),
    })

    if len(completed_records) % 1000 == 0 or len(completed_records) == n_total:
        df_cur = pd.DataFrame(completed_records)
        df_cur.to_parquet(OUTPUT_CHECKPOINT, index=False)
        elapsed = time.time() - start_time
        rate = len(completed_records) / max(elapsed, 1e-5)
        remaining_mins = (n_total - len(completed_records)) / max(rate, 1e-5) / 60.0
        print(f'[{len(completed_records)}/{n_total}] Scored | Rate: {rate:.1f} prompts/s | Elapsed: {elapsed/60:.1f}m | ETA: {remaining_mins:.1f}m')
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

print(f'[+] Inference complete! Total scored: {len(completed_records)}')


In [ ]:
# Step 7: Metrics Computation, Gate Auditing & Artifact Persistence
df_final = pd.DataFrame(completed_records)

# Save standard scores.parquet (exact 6 columns matching Perron benchmark spec)
df_clean = df_final[['instance_id', 'candidate_id', 'prompt_variant', 's', 'format_mass', 'n_prompt_tokens']]
df_clean.to_parquet(FINAL_SCORES_PARQUET, index=False)
df_clean.to_parquet(FINAL_SCORES_STANDARD, index=False)

print(f'[+] Saved {len(df_clean)} rows to {FINAL_SCORES_PARQUET} ({FINAL_SCORES_PARQUET.stat().st_size} bytes)')
print(f'[+] Saved copy to {FINAL_SCORES_STANDARD}')

# Compute task-level retrieval metrics (Hit@1, Hit@3, Hit@5, Hit@10, MRR)
task_groups = df_final.groupby('instance_id')
hit_1_list = []
hit_3_list = []
hit_5_list = []
hit_10_list = []
mrr_list = []

for iid, grp in task_groups:
    sorted_grp = grp.sort_values(by='s', ascending=False)
    gold_mask = sorted_grp['is_gold'].values > 0
    
    hit_1 = bool(np.any(gold_mask[:1]))
    hit_3 = bool(np.any(gold_mask[:3]))
    hit_5 = bool(np.any(gold_mask[:5]))
    hit_10 = bool(np.any(gold_mask[:10]))
    
    gold_indices = np.where(gold_mask)[0]
    mrr = float(1.0 / (gold_indices[0] + 1)) if len(gold_indices) > 0 else 0.0
    
    hit_1_list.append(hit_1)
    hit_3_list.append(hit_3)
    hit_5_list.append(hit_5)
    hit_10_list.append(hit_10)
    mrr_list.append(mrr)

metrics = {
    'total_candidates': int(len(df_final)),
    'total_tasks': int(len(task_groups)),
    'mean_format_mass': float(df_final['format_mass'].mean()),
    'min_format_mass': float(df_final['format_mass'].min()),
    'mean_s': float(df_final['s'].mean()),
    'std_s': float(df_final['s'].std()),
    'hit_at_1': float(np.mean(hit_1_list)),
    'hit_at_3': float(np.mean(hit_3_list)),
    'hit_at_5': float(np.mean(hit_5_list)),
    'hit_at_10': float(np.mean(hit_10_list)),
    'mrr': float(np.mean(mrr_list)),
    'device_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
    'total_runtime_seconds': float(time.time() - start_time),
}

with open(FINAL_METRICS_JSON, 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

print('[+] Summary Metrics:')
print(json.dumps(metrics, indent=2))
